# PySceneDetect Overlay Comparison

This notebook does not modify the existing image extraction notebook. It creates a separate PySceneDetect-based inference output so it can be compared against the previous adaptive frame-difference overlays.

Outputs are saved under `outputs_video_pyscenedetect_compare/`.

In [1]:
from pathlib import Path
import json

import cv2
import numpy as np
import pandas as pd
from scenedetect import SceneManager, open_video
from scenedetect.detectors import ContentDetector
from tqdm.auto import tqdm

import shotguide_batch_video_inference as base

ROOT = Path.cwd()
OUTPUT_ROOT = ROOT / 'outputs_video_pyscenedetect_compare'
OUTPUT_ROOT.mkdir(exist_ok=True)

TARGET_VIDEOS = [
    ROOT / 'videos' / '0099_DS53BHjkh5o.mp4',
]

PYSCENEDETECT_THRESHOLD = 27.0
MIN_SCENE_SEC = 0.2
NUM_FRAME_SAMPLES = 3

for path in TARGET_VIDEOS:
    assert path.exists(), path

TARGET_VIDEOS

C:\Users\user\anaconda3\envs\shotguide-baseline\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[WindowsPath('C:/Temp/deep/videos/0099_DS53BHjkh5o.mp4')]

## 1. PySceneDetect Scene Detection

In [2]:
def detect_scenes_pyscenedetect(video_path: Path, threshold=27.0, min_scene_sec=0.2):
    info = base.get_video_info(video_path)
    min_scene_len = max(1, int(info['fps'] * min_scene_sec))

    video = open_video(str(video_path))
    manager = SceneManager()
    manager.add_detector(ContentDetector(threshold=threshold, min_scene_len=min_scene_len))
    manager.detect_scenes(video=video, show_progress=False)
    scenes = manager.get_scene_list()

    if not scenes:
        scenes = [(video.base_timecode, video.duration)]

    rows = []
    frame_count = info['frame_count']
    fps = info['fps']
    for i, (start_tc, end_tc) in enumerate(scenes, start=1):
        start_frame = max(0, int(start_tc.get_frames()))
        end_frame = min(frame_count - 1, max(start_frame, int(end_tc.get_frames()) - 1))
        rows.append({
            'video_path': str(video_path.resolve()),
            'video_name': video_path.name,
            'video_id': video_path.stem.split('_')[0],
            'scene_index': i,
            'start_frame': start_frame,
            'end_frame': end_frame,
            'start_time': start_frame / fps,
            'end_time': end_frame / fps,
            'duration_sec': (end_frame - start_frame + 1) / fps,
            'threshold': threshold,
            'fps': fps,
            'detector': 'PySceneDetect ContentDetector',
        })

    return pd.DataFrame(rows), info

preview_rows = []
for video_path in TARGET_VIDEOS:
    scene_df, info = detect_scenes_pyscenedetect(video_path, PYSCENEDETECT_THRESHOLD, MIN_SCENE_SEC)
    preview_rows.append({
        'video_id': video_path.stem.split('_')[0],
        'video_name': video_path.name,
        'duration_sec': info['duration_sec'],
        'scene_count': len(scene_df),
        'avg_scene_duration_sec': scene_df['duration_sec'].mean(),
    })

pd.DataFrame(preview_rows)

C:\Users\user\AppData\Local\Temp\ipykernel_6608\335256370.py:18: DeprecationWarning: get_frames() is deprecated, use the `frame_num` property instead.
  start_frame = max(0, int(start_tc.get_frames()))
C:\Users\user\AppData\Local\Temp\ipykernel_6608\335256370.py:19: DeprecationWarning: get_frames() is deprecated, use the `frame_num` property instead.
  end_frame = min(frame_count - 1, max(start_frame, int(end_tc.get_frames()) - 1))


,video_id,video_name,duration_sec,scene_count,avg_scene_duration_sec
0,0099,0099_DS53BHjkh5o.mp4,14.766667,1,14.766667


## 2. Run CLIP Prediction and Render PySceneDetect Overlays

In [3]:
clip_model, clip_preprocess, head, idx_to_shot = base.load_models()

summary_rows = []
all_scene_rows = []

for video_path in tqdm(TARGET_VIDEOS):
    video_output_dir = OUTPUT_ROOT / video_path.stem
    frames_dir = video_output_dir / 'scene_frames'
    video_output_dir.mkdir(parents=True, exist_ok=True)

    scene_df, info = detect_scenes_pyscenedetect(video_path, PYSCENEDETECT_THRESHOLD, MIN_SCENE_SEC)
    scene_df = base.extract_scene_frames(video_path, scene_df, frames_dir, num_samples=NUM_FRAME_SAMPLES)
    scene_df.to_csv(video_output_dir / 'scene_metadata.csv', index=False, encoding='utf-8-sig')

    pred_df, embeddings = base.predict_scenes(scene_df, clip_model, clip_preprocess, head, idx_to_shot)
    pred_df.to_csv(video_output_dir / 'scene_predictions.csv', index=False, encoding='utf-8-sig')
    np.savez_compressed(video_output_dir / 'scene_clip_embeddings.npz', embeddings=embeddings)

    overlay_path = video_output_dir / f'{video_path.stem}_pyscenedetect_overlay.mp4'
    base.render_overlay_video(video_path, pred_df, overlay_path)

    summary_rows.append({
        'video_id': video_path.stem.split('_')[0],
        'video_name': video_path.name,
        'detector': 'PySceneDetect ContentDetector',
        'threshold': PYSCENEDETECT_THRESHOLD,
        'duration_sec': info['duration_sec'],
        'scene_count': len(pred_df),
        'avg_scene_duration_sec': float(pred_df['duration_sec'].mean()),
        'min_scene_duration_sec': float(pred_df['duration_sec'].min()),
        'max_scene_duration_sec': float(pred_df['duration_sec'].max()),
        'text_scene_count': int(pred_df['pred_has_text'].sum()),
        'shot_label_counts': json.dumps(pred_df['pred_shot_type'].value_counts().to_dict(), ensure_ascii=False),
        'overlay_path': str(overlay_path.resolve()),
    })
    all_scene_rows.append(pred_df)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_ROOT / 'pyscenedetect_batch_summary.csv', index=False, encoding='utf-8-sig')
pd.concat(all_scene_rows, ignore_index=True).to_csv(OUTPUT_ROOT / 'pyscenedetect_scene_predictions.csv', index=False, encoding='utf-8-sig')

summary_df

C:\Users\user\anaconda3\envs\shotguide-baseline\lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


  0%|          | 0/1 [00:00<?, ?it/s]

INFO:pyscenedetect:Detecting scenes...


C:\Users\user\AppData\Local\Temp\ipykernel_6608\335256370.py:18: DeprecationWarning: get_frames() is deprecated, use the `frame_num` property instead.
  start_frame = max(0, int(start_tc.get_frames()))
C:\Users\user\AppData\Local\Temp\ipykernel_6608\335256370.py:19: DeprecationWarning: get_frames() is deprecated, use the `frame_num` property instead.
  end_frame = min(frame_count - 1, max(start_frame, int(end_tc.get_frames()) - 1))


100%|██████████| 1/1 [00:09<00:00,  9.43s/it]

100%|██████████| 1/1 [00:09<00:00,  9.43s/it]

,video_id,video_name,detector,threshold,duration_sec,scene_count,avg_scene_duration_sec,min_scene_duration_sec,max_scene_duration_sec,text_scene_count,shot_label_counts,overlay_path
0,0099,0099_DS53BHjkh5o.mp4,PySceneDetect ContentDetector,27.0,14.766667,1,14.766667,14.766667,14.766667,0,"{""medium"": 1}",C:\Temp\deep\outputs_video_pyscenedetect_compa...


## 3. Compare with Previous Frame-diff Output

In [4]:
old_rows = []
for video_path in TARGET_VIDEOS:
    old_csv = ROOT / 'outputs_video_batch_test' / video_path.stem / 'scene_predictions.csv'
    if old_csv.exists():
        old_df = pd.read_csv(old_csv)
        old_rows.append({
            'video_id': video_path.stem.split('_')[0],
            'video_name': video_path.name,
            'detector': 'Previous adaptive frame-diff',
            'scene_count': len(old_df),
            'avg_scene_duration_sec': old_df['duration_sec'].mean(),
            'text_scene_count': int(old_df['pred_has_text'].sum()),
            'shot_label_counts': json.dumps(old_df['pred_shot_type'].value_counts().to_dict(), ensure_ascii=False),
            'overlay_path': str((ROOT / 'outputs_video_batch_test' / video_path.stem / f'{video_path.stem}_overlay.mp4').resolve()),
        })

comparison_df = pd.concat([pd.DataFrame(old_rows), summary_df], ignore_index=True)
comparison_df.to_csv(OUTPUT_ROOT / 'detector_comparison_summary.csv', index=False, encoding='utf-8-sig')
comparison_df[['video_id', 'detector', 'scene_count', 'avg_scene_duration_sec', 'text_scene_count', 'shot_label_counts', 'overlay_path']]

,video_id,detector,scene_count,avg_scene_duration_sec,text_scene_count,shot_label_counts,overlay_path
0,0099,Previous adaptive frame-diff,3,4.922222,2,"{""object"": 2, ""space"": 1}",C:\Temp\deep\outputs_video_batch_test\0099_DS5...
1,0099,PySceneDetect ContentDetector,1,14.766667,0,"{""medium"": 1}",C:\Temp\deep\outputs_video_pyscenedetect_compa...
